In [1]:
import pandas as pd 
import re
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

c:\Users\YOGA\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data = pd.read_csv('../data/samsum-train.csv')
val_data = pd.read_csv('../data/samsum-validation.csv')

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 14732 entries, 0 to 14731
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        14732 non-null  str  
 1   dialogue  14731 non-null  str  
 2   summary   14732 non-null  str  
dtypes: str(3)
memory usage: 9.2 MB


# Data Preprocessing

In [6]:
train_data = train_data.dropna()

In [7]:
train_data.shape

(14731, 3)

In [8]:
val_data.shape

(818, 3)

In [9]:
def clean_data(text):
    text = re.sub(r'r/n/',' ',text)
    text = re.sub(r's+',' ',text)
    text = re.sub(r'<.*?>',' ',text)

    return text

In [10]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

# Tokenization

In [11]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

In [12]:
def tokenize(data):
    inputs = tokenizer(data['dialogue'],padding='max_length',max_length=512,truncation=True)
    target = tokenizer(data['summary'],padding='max_length',max_length=150,truncation=True)
    inputs['labels'] = target['input_ids']

    return inputs

In [14]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [ ]:
train_dataset[0] 
# input_ids => tokenized dialogue data
# attention_mask => 1 for real data 0 for padded data
# labels => input_ids of tokenized summary data

{'input_ids': [21542, 10, 27, 13635, 7364, 3, 5, 531, 25, 241, 3, 7159, 58, 16637, 10, 10625, 55, 21542, 10, 27, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

# Model Building

In [18]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 2798.10it/s]


In [20]:
training_args = TrainingArguments(
    output_dir='./model',
    num_train_epochs= 4,

    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy='epoch',
    save_strategy='epoch',

    warmup_steps=500
)

In [21]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [22]:
trainer.train()

c:\Users\YOGA\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 